# 05 — Endpoint Deployment & Scoring

## Objective
Deploy the best AutoML model as a real-time endpoint and generate
90-day forecasts that feed into the Power BI dashboard.

**Pipeline:**
1. Register the best model from AutoML
2. Create a managed online endpoint
3. Deploy the model to the endpoint
4. Score the next 90 days of predictions
5. Export `fact_forecasts` for Power BI

This demonstrates the full ML lifecycle — training → deployment → serving.

In [ ]:
# === Setup ===
import os, sys, json
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient
import pandas as pd
from pathlib import Path

# Load job info
with open('../data/output/automl_job_info.json') as f:
    job_info = json.load(f)
JOB_NAME = job_info['job_name']

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=os.getenv('AZURE_SUBSCRIPTION_ID'),
    resource_group_name=os.getenv('AZURE_RESOURCE_GROUP'),
    workspace_name=os.getenv('AML_WORKSPACE_NAME'),
)
print(f'✓ Connected | Job: {JOB_NAME}')

## 1. Deploy the Model

Using our deployment script, which handles:
- Finding the best model from the experiment
- Registering it as a versioned model asset
- Creating a managed endpoint with API key auth
- Deploying with the right VM size

In [ ]:
# Run the deployment script
!cd .. && python src/deploy_endpoint.py --job-name {JOB_NAME}

## 2. Generate 90-Day Forecasts

Score all store × product family combinations for the next 90 days.
This creates the `fact_forecasts` table for Power BI.

In [ ]:
# Score forecasts against the deployed endpoint
# Use --demo flag to generate synthetic forecasts if endpoint isn't ready
!cd .. && python src/score_forecasts.py --demo

In [ ]:
# Verify the output
fact_forecasts = pd.read_csv('../data/processed/fact_forecasts.csv')

print(f'Forecast table shape: {fact_forecasts.shape}')
print(f'\nDate range: {fact_forecasts["date"].min()} to {fact_forecasts["date"].max()}')
print(f'\nHorizon breakdown:')
print(fact_forecasts['forecast_horizon'].value_counts())
print(f'\nTotal predicted revenue: ${fact_forecasts["predicted_revenue"].sum():,.0f}')

fact_forecasts.head()

## 3. Data Ready for Power BI

The following files are now in `data/processed/` ready for Power BI import:

| File | Description | Power BI Table |
|------|-------------|----------------|
| `dim_date.csv` | Date dimension | dim_date |
| `dim_store.csv` | Store dimension (with regions for RLS) | dim_store |
| `dim_product.csv` | Product family dimension | dim_product |
| `fact_sales.csv` | Historical actuals | fact_sales |
| `fact_forecasts.csv` | 90-day predictions | fact_forecasts |
| `fact_forecasts_weekly.csv` | Weekly aggregated forecasts | fact_forecasts_weekly |

**Next step:** Open Power BI Desktop and follow `powerbi/data_model.md` for import instructions.

## 4. Cleanup (Optional)

**Important:** Delete the endpoint when you're done to avoid ongoing charges.
Managed endpoints charge per-hour while deployed, even with no traffic.

In [ ]:
# UNCOMMENT to delete endpoint (saves money when not in use)
# ml_client.online_endpoints.begin_delete(
#     name=os.getenv('ENDPOINT_NAME', 'sales-forecast-endpoint')
# ).result()
# print('✓ Endpoint deleted — no more charges.')

print('Remember to delete the endpoint when done to avoid charges!')
print(f'  az ml online-endpoint delete --name {os.getenv("ENDPOINT_NAME", "sales-forecast-endpoint")} -y')